In [ ]:
from pathlib import Path
import time
import numpy as np
import matplotlib.pyplot as plt
from tensorflow import keras
from tensorflow.keras.datasets import fashion_mnist


from data import load_cv_data_fashion_mnist, CLASS_NAMES
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    log_loss,
    accuracy_score,
    classification_report,
    ConfusionMatrixDisplay,
    RocCurveDisplay,
)

project_root = Path.cwd().parent if Path.cwd().name == "code" else Path.cwd()
figures_dir = project_root / "figures"
figures_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
(Xtr, ytr), (Xtest, ytest) = load_cv_data_fashion_mnist()

In [ ]:
C_values = [0.001, 0.01, 0.1, 1.0]
cv_results = {}
cv = StratifiedKFold(5, shuffle=True, random_state=2026)

for C in C_values:
    fold_losses = []
    fold_accuracies = []

    for train_idx, val_idx in cv.split(Xtr, ytr):
        X_fold_train = Xtr[train_idx]
        X_fold_val = Xtr[val_idx]
        mean, std = X_fold_train.mean(), X_fold_train.std()
        X_fold_train = ((X_fold_train - mean) / std).astype("float32").reshape(len(train_idx), 784)
        X_fold_val = ((X_fold_val - mean) / std).astype("float32").reshape(len(val_idx), 784)

        model_cv = LogisticRegression(
            solver="lbfgs",
            max_iter=1000,
            random_state=2026,
            C=C
        )
        model_cv.fit(X_fold_train, ytr[train_idx])
        val_probs = model_cv.predict_proba(X_fold_val)
        val_pred = model_cv.predict(X_fold_val)
        fold_losses.append(log_loss(ytr[val_idx], val_probs, labels=np.arange(10)))
        fold_accuracies.append(accuracy_score(ytr[val_idx], val_pred))

    cv_results[C] = {
        "loss": np.mean(fold_losses),
        "accuracy": np.mean(fold_accuracies)
    }

best_C = min(cv_results, key=lambda C: cv_results[C]["loss"])

print("C | mean val loss | mean val accuracy")
for C, result in cv_results.items():
    print(f"{C:<6g} | {result['loss']:.4f}        | {result['accuracy']:.4f}")
print(f"Best C: {best_C}")

In [ ]:
# Load all 60,000 training and 10,000 test images
(x_train, y_train), (x_test, y_test) = fashion_mnist.load_data()

# Standardize using training statistics only
mean, std = x_train.mean(), x_train.std()
X_train = ((x_train - mean) / std).astype("float32").reshape(len(x_train), -1)
X_test = ((x_test - mean) / std).astype("float32").reshape(len(x_test), -1)

model = LogisticRegression(
    solver="lbfgs",
    max_iter=1000,
    random_state=2026,
    C=best_C
)
start_time = time.perf_counter()
model.fit(X_train, y_train)
training_time = time.perf_counter() - start_time
n_params = model.coef_.size + model.intercept_.size

In [ ]:
# Evaluate on the test set once
probs = model.predict_proba(X_test)
y_pred = model.predict(X_test)

acc = accuracy_score(y_test, y_pred)
se = np.sqrt(acc * (1 - acc) / len(y_test))
print(f"Test accuracy: {acc:.4f} ? {se:.4f}")
print(f"Training time: {training_time:.2f} seconds")
print(f"Number of parameters: {n_params}")
print(classification_report(
    y_test,
    y_pred,
    target_names=CLASS_NAMES,
    digits=4
))

fig, ax = plt.subplots(1, 2, figsize=(15, 6))
ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred,
    display_labels=CLASS_NAMES,
    ax=ax[0],
    xticks_rotation=90,
    cmap="Blues"
)
ax[0].set_title("Confusion matrix")

for i, name in enumerate(CLASS_NAMES):
    RocCurveDisplay.from_predictions(
        (y_test == i).astype(int),
        probs[:, i],
        name=name,
        ax=ax[1]
    )
ax[1].set_title("One-vs-rest ROC curves")
ax[1].set_xlabel("False Positive Rate")
ax[1].set_ylabel("True Positive Rate")
ax[1].legend(fontsize=8)

plt.tight_layout()
fig.savefig(
    figures_dir / "LogReg_CV_final_results.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()